# Step 2: Problem Definition & Use Case Specification

## 1. Problem Statement
Phishing attacks deceive internet users into exposing sensitive credentials, financial details, and personal data by disguising malicious websites as trustworthy platforms. Detecting phishing URLs before a user visits them is a critical first-line security defense.

## 2. System Scope & Intended Users
- **Intended Users**: End users, security administrators, and web application clients.
- **Input**: A raw URL string submitted through the Streamlit web application (e.g., `https://login.example-bank.com`).
- **Output**: Classification as **Phishing** (`is_phishing = 1`) or **Legitimate** (`is_phishing = 0`), along with model probability and risk guidance.

## 3. System Constraints & Operating Rules
- **Model Restrictions**: Permitted algorithms are strictly limited to **Logistic Regression** and **Decision Tree Classifier**.
- **No Live Web Fetching**: The web app strictly extracts deterministic features from the URL string itself (`FS-URL`). No web scraping, DOM parsing, or HTTP network requests are executed at inference time.
- **Safety Disclaimer**: The application provides a statistical prediction, NOT an absolute guarantee of website safety.

## 4. Target Mapping & Cost Matrix
- **Original Labels**: `label = 0` (Phishing), `label = 1` (Legitimate).
- **Derived Project Target**: `is_phishing = (label == 0).astype(int)`
  - `is_phishing = 1`: **Phishing** (Positive Class)
  - `is_phishing = 0`: **Legitimate** (Negative Class)

### Cost Breakdown of Errors:
- **False Negative (FN)** [HIGH COST]: Predicting a phishing site as Legitimate. User visits malicious site -> credential theft / security breach.
- **False Positive (FP)** [MODERATE COST]: Predicting a legitimate site as Phishing. User experiences unnecessary warning -> mild friction.
- **Primary Metric Focus**: Maximize **Phishing Recall** and **F1-Score / PR-AUC** to aggressively reduce False Negatives.

In [5]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add project root to sys.path
sys.path.append("..")
from src.config import RAW_DATA_PATH, RAW_TARGET_COL, DERIVED_TARGET_COL

# Load raw data and verify target mapping
df = pd.read_csv(RAW_DATA_PATH)
df[DERIVED_TARGET_COL] = (df[RAW_TARGET_COL] == 0).astype("int8")

print(f"Dataset total records: {len(df):,}")
print("\nTarget Mapping Breakdown:")
target_counts = df[DERIVED_TARGET_COL].value_counts()
target_pcts = df[DERIVED_TARGET_COL].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    "Class Name": ["Phishing (is_phishing=1)", "Legitimate (is_phishing=0)"],
    "Count": [target_counts[1], target_counts[0]],
    "Percentage": [f"{target_pcts[1]:.2f}%", f"{target_pcts[0]:.2f}%"]
})
print(summary_df.to_string(index=False))


Dataset total records: 30,000

Target Mapping Breakdown:
                Class Name  Count Percentage
  Phishing (is_phishing=1)  12843     42.81%
Legitimate (is_phishing=0)  17157     57.19%


## Step 3: Data Acquisition and Understanding

### 1. Dataset Source & Provenance
- **Dataset Name**: PhiUSIIL Phishing URL Dataset (30,000 sample subset)
- **Authors/Citation**: Arvind Prasad & Shalini Chandra (2023), *PhiUSIIL: A diverse and comprehensive dataset for phishing URL detection*, Cybersecurity (Springer), 6(1), 1-19.
- **Repository**: UCI Machine Learning Repository (Dataset ID: 967)
- **License**: Creative Commons Attribution 4.0 International (CC BY 4.0)
- **Local Workspace File**: `PhiUSIIL_Phishing_30000.csv` (7,542,406 bytes, verified in workspace root)

### 2. Dataset Overview & Dimensions
- **Total Observations**: 30,000 records
- **Total Attributes**: 56 columns
- **Missing Values**: 0 nulls across all 56 columns
- **Primary Key / Identifier**: `FILENAME` (30,000 unique string IDs)
- **Raw URL Column**: `URL` (29,989 unique URLs, 11 duplicate URLs)

### 3. Feature Taxonomy & Groupings
1. **Identifier (1)**: `FILENAME`
2. **Raw Strings (4)**: `URL`, `Domain`, `TLD`, `Title`
3. **URL Structural & Lexical Features (19)**: `URLLength`, `DomainLength`, `IsDomainIP`, `TLDLength`, `NoOfSubDomain`, `HasObfuscation`, `NoOfObfuscatedChar`, `ObfuscationRatio`, `NoOfLettersInURL`, `LetterRatioInURL`, `NoOfDegitsInURL`, `DegitRatioInURL`, `NoOfEqualsInURL`, `NoOfQMarkInURL`, `NoOfAmpersandInURL`, `NoOfOtherSpecialCharsInURL`, `SpacialCharRatioInURL`, `IsHTTPS`
4. **URL Statistical Probabilities (4)**: `URLSimilarityIndex`, `CharContinuationRate`, `TLDLegitimateProb`, `URLCharProb`
5. **Page Content / HTML Features (27)**: `LineOfCode`, `LargestLineLength`, `HasTitle`, `DomainTitleMatchScore`, `URLTitleMatchScore`, `HasFavicon`, `Robots`, `IsResponsive`, `NoOfURLRedirect`, `NoOfSelfRedirect`, `HasDescription`, `NoOfPopup`, `NoOfiFrame`, `HasExternalFormSubmit`, `HasSocialNet`, `HasSubmitButton`, `HasHiddenFields`, `HasPasswordField`, `Bank`, `Pay`, `Crypto`, `HasCopyrightInfo`, `NoOfImage`, `NoOfCSS`, `NoOfJS`, `NoOfSelfRef`, `NoOfEmptyRef`, `NoOfExternalRef`
6. **Target Column (1)**: `label` (`0` = Phishing, `1` = Legitimate)

In [6]:
import pandas as pd
import sys
sys.path.append("..")
from src.config import RAW_DATA_PATH

df = pd.read_csv(RAW_DATA_PATH)

print("=== DATA AUDIT SUMMARY ===")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Total Missing Values: {df.isnull().sum().sum()}")
print(f"Unique FILENAME count: {df['FILENAME'].nunique()}")
print(f"Unique URL count: {df['URL'].nunique()}")

# Sample data view across feature types
sample_cols = ['FILENAME', 'URL', 'URLLength', 'IsDomainIP', 'IsHTTPS', 'LineOfCode', 'label']
print("\nSample Data across Feature Categories:")
print(df[sample_cols].head(3).to_string(index=False))


=== DATA AUDIT SUMMARY ===
Rows: 30,000
Columns: 56
Total Missing Values: 0
Unique FILENAME count: 30000
Unique URL count: 29989

Sample Data across Feature Categories:
    FILENAME                                  URL  URLLength  IsDomainIP  IsHTTPS  LineOfCode  label
  683936.txt https://www.westannouncements.com.au         35           0        1       11716      1
 mw29982.txt          http://www.f0585194.xsph.ru         26           0        0           9      0
mw204791.txt            http://www.buthagnere.com         24           0        0           2      0


## Step 4: Data Preparation and Quality Audit

### 1. Data Quality Audit Findings
- **Raw CSV Record Count**: 30,000 rows, 56 columns (Preserved 100% unchanged in root).
- **Missing Values**: 0 null/missing cells across all 56 attributes.
- **Complete Row Duplicates**: 0 complete duplicate rows across all 56 features.
- **FILENAME Duplicates**: 0 duplicate `FILENAME` values (30,000 unique records).
- **URL Duplicates Inspection**:
  - 11 unique URLs appear twice (22 rows total).
  - **Label Consistency**: All 11 duplicate URL pairs have **100% CONSISTENT labels** (`label = 0` / Phishing for all 11 URL pairs). Zero conflicting URL labels found.
- **Domain Duplicates & Conflicting Domain Inspection**:
  - 1,063 unique domains appear multiple times (1,311 rows total).
  - 1 domain (`www.mod.gov.in`) has URLs with differing scheme/labels (`http://...` vs `https://...`). This is legitimate behavior as scheme (`IsHTTPS`) differentiates encryption state.
- **Value Boundary Integrity**:
  - Target `label`: Only `{0, 1}` present (Valid).
  - All 17 Binary Features: Only `{0, 1}` present (Valid).
  - All 19 Count Features: Zero negative values found (Valid).

### 2. Data Cleaning & Handling Decisions
1. **Preservation**: The raw CSV `PhiUSIIL_Phishing_30000.csv` is preserved unchanged.
2. **No Data Dropping at Step 4**: Per project rules, zero rows are removed during raw audit.
3. **Split Leakage Prevention Strategy**: In Step 9 (data splitting), URL deduplication or Stratified Group splitting will be used to ensure identical URL instances do not leak across training and test splits.

In [7]:
from src.data_validation import audit_data_quality, add_derived_target
import pandas as pd
from src.config import RAW_DATA_PATH

df_raw = pd.read_csv(RAW_DATA_PATH)
audit_results = audit_data_quality(df_raw)

print("=== DATA QUALITY AUDIT RESULTS ===")
for k, v in audit_results.items():
    print(f"{k}: {v}")


=== DATA QUALITY AUDIT RESULTS ===
total_rows: 30000
total_columns: 56
missing_values_count: 0
complete_duplicate_rows: 0
duplicate_filenames: 0
duplicate_urls: 11
conflicting_url_labels_count: 0
unique_raw_labels: [0, 1]
labels_valid: True
invalid_binary_columns: {}


## Step 5: Preprocessing and Leakage Audit

### 1. Data Leakage Audit Findings (`FILENAME` & Source Artifacts)
- **Critical Leakage Feature Detected**: `FILENAME` contains collection source prefixes:
  - `mw` prefix (4,681 records): **100% Phishing** (`is_phishing = 1`)
  - `op` prefix (1,168 records): **100% Phishing** (`is_phishing = 1`)
  - Numeric prefixes (e.g. `10...` to `99...`): **100% Legitimate** (`is_phishing = 0`)
- **Mandatory Action**: `FILENAME` is an explicit target proxy caused by data collection artifacts. It is added to `EXCLUDED_FEATURES` and **strictly excluded from all model training** to prevent artificial 100% accuracy leakage.

### 2. `IsHTTPS` Feature Audit
- `IsHTTPS = 0` (HTTP connection): 100% Phishing in this dataset (7,370 records).
- `IsHTTPS = 1` (HTTPS connection): 73.27% Legitimate, 26.73% Phishing.
- **Verdict**: `IsHTTPS` is a valid, deterministic URL structural feature derivable from raw URL strings (`url.lower().startswith('https://')`).

### 3. Outlier Audit (IQR Bounds Analysis)
- **URLLength**: Range 13 to 4,274 chars. 2,893 records (9.64%) exceed $Q3 + 1.5 \times IQR$.
- **NoOfDegitsInURL**: Range 0 to 1,912 digits. 6,490 records (21.63%) exceed $Q3 + 1.5 \times IQR$.
- **Outlier Removal Decision**: **Do NOT remove extreme observations.** Extreme URL lengths and high digit counts represent authentic phishing obfuscation techniques. Truncating or deleting them would erode genuine phishing patterns.

### 4. Preprocessing & Transformation Architecture
- **Scaling Strategy**: Continuous numeric features are scaled using `StandardScaler` (or `RobustScaler`). Binary features pass through unchanged.
- **Zero Leakage Rule**: All scalers and transformers are wrapped inside `sklearn.pipeline.Pipeline` and `ColumnTransformer`, ensuring fitting occurs **strictly on training folds** during cross-validation.

In [8]:
from src.preprocessing import EXCLUDED_FEATURES, get_feature_preprocessor, create_modeling_pipeline
from sklearn.linear_model import LogisticRegression
import pandas as pd
from src.config import RAW_DATA_PATH

df = pd.read_csv(RAW_DATA_PATH)
print("Excluded features from predictors:")
print(EXCLUDED_FEATURES)


Excluded features from predictors:
['FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label', 'is_phishing']
